# Evela Health Model - End-to-end example

This notebook walks through subscribing to the Evela Health Model, deploying it to a SageMaker endpoint, invoking it with a wearable activity summary, parsing the structured readout, and cleaning up when done.

The model is a compact (~0.5B parameter) instruction-tuned text model that converts a 30-day summary of wearable and phone activity data into a structured health dimension readout. It scores two dimensions directly from sensor data (exercise, resilience) and explicitly refuses to score the six dimensions where sensor data carries no signal.

**Intended use:** wellness and activity summarization for health and fitness applications. The model is not a medical device and must not be used for diagnosis, treatment, or clinical decision making.

**Runtime cost:** billed per endpoint-hour. Delete the endpoint when you are finished to stop billing. There is no scale-to-zero.

## Prerequisites

1. An AWS account with an active subscription to this model on AWS Marketplace or Bedrock Marketplace.
2. Service quota for the chosen inference instance type in your target region. Default: `ml.g5.xlarge`.
3. An IAM role with `AmazonSageMakerFullAccess` or an equivalent custom policy covering `sagemaker:CreateEndpoint`, `sagemaker:InvokeEndpoint`, and `sagemaker:DeleteEndpoint`.
4. `boto3` installed in this notebook kernel.

In [ ]:
import json
import time
import boto3

# ----- Fill in before running -----
REGION = "eu-central-1"
MODEL_PACKAGE_ARN = "arn:aws:sagemaker:eu-central-1:975049962758:model-package/EvelaHealth10DimensionModel"
SAGEMAKER_EXECUTION_ROLE = "arn:aws:iam::<your-account>:role/service-role/AmazonSageMaker-ExecutionRole-XXXXXXXXXX"
ENDPOINT_NAME = "evela-health-example"
INSTANCE_TYPE = "ml.g5.xlarge"
# ----------------------------------

sagemaker = boto3.client("sagemaker", region_name=REGION)
sagemaker_runtime = boto3.client("sagemaker-runtime", region_name=REGION)

## Step 1. Deploy the model to an endpoint

This creates a model, endpoint config, and endpoint from the subscribed model package. Endpoint creation takes 10 to 15 minutes.

In [ ]:
model_name = ENDPOINT_NAME + "-model"
config_name = ENDPOINT_NAME + "-config"

sagemaker.create_model(
    ModelName=model_name,
    ExecutionRoleArn=SAGEMAKER_EXECUTION_ROLE,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    EnableNetworkIsolation=True,
)

sagemaker.create_endpoint_config(
    EndpointConfigName=config_name,
    ProductionVariants=[
        {
            "VariantName": "primary",
            "ModelName": model_name,
            "InstanceType": INSTANCE_TYPE,
            "InitialInstanceCount": 1,
        }
    ],
)

sagemaker.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=config_name,
)

print(f"Endpoint {ENDPOINT_NAME} creation started. Waiting for InService status...")
waiter = sagemaker.get_waiter("endpoint_in_service")
waiter.wait(EndpointName=ENDPOINT_NAME, WaiterConfig={"Delay": 30, "MaxAttempts": 40})
print("Endpoint is InService. Billing has started.")

## Step 2. The system prompt

The system prompt below is required for correct refusal behavior. Send it unchanged on every request. The model's refusal rules depend on it.

In [ ]:
SYSTEM_PROMPT = """You are a health assessment model developed by Evela. You analyze wearable and phone activity data and produce structured health dimension assessments.

You MUST follow these rules:
1. Only assess dimensions for which the input data provides direct signal.
2. For exercise: use daily step counts, distance, and floors climbed to assign a level from 1 (very low) to 5 (very high).
3. For resilience: use resting heart rate as a limited proxy, only when heart rate data is available. State clearly that this is an incomplete assessment.
4. For all other dimensions (nutrition, sleep, prevention, sexual health, skin and hair, social connections): explicitly state that the dimension cannot be assessed from the available sensor data, and explain why.
5. Never invent or hallucinate data that is not present in the input.
6. This is a wellness tool, not a medical device. Do not make diagnostic claims."""

## Step 3. Build a user message from raw samples

The model expects a plain-text summary of a 30-day window, not raw per-minute sensor readings. The helper below takes typical HealthKit or Health Connect style daily samples and produces the exact format the model expects.

Replace the example `raw_samples` with your own data when integrating.

In [ ]:
from statistics import mean, quantiles
from typing import Optional

def build_user_message(
    days_with_data: int,
    sources: list[str],
    daily_steps: Optional[list[float]] = None,
    daily_distance_m: Optional[list[float]] = None,
    daily_floors: Optional[list[float]] = None,
    daily_resting_hr: Optional[list[float]] = None,
    daily_mean_hr: Optional[list[float]] = None,
    daily_peak_hr: Optional[list[float]] = None,
    window_days: int = 30,
) -> str:
    def _avg(x):
        return round(mean(x)) if x else None

    def _iqr(x):
        if not x or len(x) < 4:
            return None
        q = quantiles(x, n=4)
        return (round(q[0]), round(q[2]))

    coverage_pct = round(100 * days_with_data / window_days)
    has_hr = bool(daily_resting_hr or daily_mean_hr or daily_peak_hr)

    lines = [
        f"Period: {window_days}-day window",
        f"Days with data: {days_with_data} of {window_days} ({coverage_pct}% coverage)",
        f"Data sources: {', '.join(sources)}",
        f"Heart rate available: {'yes' if has_hr else 'no'}",
        "",
        "Daily averages:",
    ]
    if daily_steps is not None:
        lines.append(f"  Steps: {_avg(daily_steps)} steps")
    if daily_distance_m is not None:
        lines.append(f"  Distance: {_avg(daily_distance_m)} meters")
    if daily_floors is not None:
        lines.append(f"  Floors climbed: {round(mean(daily_floors), 1)} floors")
    iqr = _iqr(daily_steps) if daily_steps else None
    if iqr:
        lines.append(f"  Steps interquartile range: {iqr[0]} - {iqr[1]}")

    if has_hr:
        lines += ["", "Heart rate:"]
        if daily_resting_hr:
            lines.append(f"  Resting (estimated): {_avg(daily_resting_hr)} bpm")
        if daily_mean_hr:
            lines.append(f"  Daily average: {_avg(daily_mean_hr)} bpm")
        if daily_peak_hr:
            lines.append(f"  Peak: {_avg(daily_peak_hr)} bpm")

    return "\n".join(lines)


# Example: an active user with an Oura ring, 28 of 30 days of data
user_message = build_user_message(
    days_with_data=28,
    sources=["device", "oura"],
    daily_steps=[9100, 8800, 10200, 11500, 7900, 9400, 10800] * 4,
    daily_distance_m=[6500, 6300, 7200, 8100, 5600, 6700, 7700] * 4,
    daily_floors=[11.0, 12.5, 13.0, 14.5, 10.5, 12.0, 13.5] * 4,
    daily_resting_hr=[57, 58, 58, 59, 56, 58, 59] * 4,
    daily_mean_hr=[73, 74, 75, 76, 72, 74, 75] * 4,
    daily_peak_hr=[152, 158, 162, 150, 148, 160, 165] * 4,
)

print(user_message)

## Step 4. Invoke the endpoint

In [ ]:
request = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
}

response = sagemaker_runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps(request),
)

payload = json.loads(response["Body"].read())
print(payload["output"])
print()
print(f"Processing time: {payload['processing_time_ms']} ms")

## Step 5. Parse the structured readout

The output text has a stable structure. The helper below extracts the per-dimension level for each dimension, returning `None` for refused ones.

In [ ]:
import re

DIMENSIONS = [
    "exercise",
    "resilience",
    "nutrition",
    "sleep",
    "prevention",
    "sexual_health",
    "skin_hair",
    "social_connections",
]

def parse_readout(text: str) -> dict:
    out = {}
    for dim in DIMENSIONS:
        pattern = rf"^{dim}:\s*(\d)/5"
        m = re.search(pattern, text, re.MULTILINE)
        out[dim] = int(m.group(1)) if m else None
    return out


parsed = parse_readout(payload["output"])
for dim, level in parsed.items():
    status = f"{level}/5" if level is not None else "refused"
    print(f"  {dim:<22} {status}")

## Step 6. A second example: phone-only user, no heart rate

This shows the refusal behavior for resilience when heart rate data is absent.

In [ ]:
user_message_phone_only = build_user_message(
    days_with_data=15,
    sources=["device"],
    daily_steps=[2100, 1800, 2400, 2500, 1900, 2200, 2300] * 2,
    daily_distance_m=[1400, 1200, 1600, 1700, 1300, 1500, 1550] * 2,
    daily_floors=[1.5, 1.0, 2.0, 1.5, 1.0, 1.5, 1.5] * 2,
)

response2 = sagemaker_runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="application/json",
    Accept="application/json",
    Body=json.dumps({
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_message_phone_only},
        ]
    }),
)

payload2 = json.loads(response2["Body"].read())
print(payload2["output"])

## Step 7. Clean up

Delete the endpoint, endpoint config, and model to stop billing. The subscription remains active and can be redeployed later without resubscribing.

In [ ]:
sagemaker.delete_endpoint(EndpointName=ENDPOINT_NAME)
sagemaker.delete_endpoint_config(EndpointConfigName=config_name)
sagemaker.delete_model(ModelName=model_name)
print("Endpoint, endpoint config, and model deleted. Billing has stopped.")

## Troubleshooting

**Deployment fails with `CapacityError`.** AWS has no instances of the requested type in your region at that moment. Retry 15 to 60 minutes later, or temporarily try a different supported instance type.

**Deployment fails with `Resource limit exceeded`.** Your account quota for the instance type is zero or insufficient. Request an increase under Service Quotas: `ml.g5.xlarge for endpoint usage`.

**Invocation returns 503 `Model not ready`.** The endpoint is still loading. The model takes 30 to 90 seconds to load after the endpoint reaches `InService`. Retry after a short delay.

**Invocation returns 400 `Invalid JSON`.** The request body is not valid JSON, or `messages` is missing. Check the request shape in Step 4.

**Model scores a dimension it should refuse.** Verify the system prompt is sent unchanged. The refusal rules live in the system prompt; without it, the model's behavior is undefined.

**Model refuses `resilience` on data that has heart rate.** Confirm the user message literally contains `Heart rate available: yes` and a `Heart rate:` section. The model keys off those strings.